## 02c · Perfilamiento de bronce (evidencia)
Reproduce la sección 2 de la Entrega 1 sobre el bronce cargado por streaming en la Entrega 2. Los conteos coinciden con los de la E1 (49,897 MMSI no estándar; 159,987 / 10.29M / 33.5M sentinels; 12 COG inválidos; 1,388 duplicados exactos y 284 conflictos de llave), lo que confirma que la ingesta nueva no alteró los datos. No forma parte del job diario.

In [0]:
%run ./00_config

In [0]:
from pyspark.sql import functions as F

b = spark.table(T("bronze", "ais_raw"))
cols = ['MMSI','BaseDateTime','LAT','LON','SOG','COG','Heading','VesselName','IMO','CallSign',
        'VesselType','Status','Length','Width','Draft','Cargo','TransceiverClass']

lat, lon = F.col("LAT").cast("double"), F.col("LON").cast("double")
sog, cog, hdg = F.col("SOG").cast("double"), F.col("COG").cast("double"), F.col("Heading").cast("double")
ts = F.to_timestamp("BaseDateTime")

checks = {
    "total":                    F.lit(True),
    "mmsi_no_9_digitos":        ~F.col("MMSI").rlike(r"^\d{9}$") | F.col("MMSI").isNull(),
    "ts_invalido":              ts.isNull(),
    "lat_nula_o_no_numerica":   lat.isNull(),
    "lat_fuera_rango":          (lat < -90) | (lat > 90),
    "lon_nula_o_no_numerica":   lon.isNull(),
    "lon_fuera_rango":          (lon < -180) | (lon > 180),
    "sog_nula":                 sog.isNull(),
    "sog_negativa":             sog < 0,
    "sog_sentinel_102_3":       F.abs(sog - 102.3) < 0.05,
    "sog_50_a_102":             (sog > 50) & (sog < 102.25),
    "cog_sentinel_360":         F.abs(cog - 360.0) < 0.05,
    "cog_fuera_rango":          (cog < 0) | (cog > 360.05),
    "heading_sentinel_511":     F.abs(hdg - 511) < 0.5,
    "imo_nulo":                 F.col("IMO").isNull(),
    "imo_mal_formado":          F.col("IMO").isNotNull() & ~F.col("IMO").rlike(r"^IMO\d{7}$"),
}
r = b.agg(*[F.sum(c.cast("int")).alias(k) for k, c in checks.items()]).first().asDict()
tot = r["total"]
for k, v in r.items():
    print(f"{k:26s} {v:>12,}  {100*v/tot:6.3f}%")

In [0]:
n_key   = b.select("MMSI", "BaseDateTime").distinct().count()
n_exact = b.select(*cols).distinct().count()
print(f"total:                         {tot:,}")
print(f"dup por (MMSI, BaseDateTime):  {tot - n_key:,}")
print(f"dup exactos (todas las cols):  {tot - n_exact:,}")